# Quiz 4 — Open-Note Survival Guide
### DATASCI Python: pandas (A–C) → Object-Oriented Programming (D–F)

**How to use during the quiz:** Find the letter matching the question. Read **WHAT TO DO**, copy the closest **TEMPLATE**, replace names/columns/conditions, and run the **CHECK**. All practice examples below are self-contained; the quiz may provide different filenames or classes.

> This notebook follows the **uploaded `practice_quiz_4-solutions.ipynb`** in question order. The examples are adapted for learning; check the quiz's actual requirements before copying. The original practice solution's `.iloc[28:33, 4:]` relies on the specific column order of its `features.csv`.

**Navigation:** A Random subsets · B `.iloc` and sorting · C `.apply` and new columns · D Mutating objects · E Inheritance · F Objects holding objects and pairing · Final checklist.

## A — Split a DataFrame into random groups
**Question pattern:** Read a CSV, count rows, generate one random value per row, create two subsets with `.query()`, calculate their percentages.

### WHAT TO DO
1. Import `pandas as pd`, `numpy as np`; read the **actual** filename.
2. Set `n = len(df)` (or `df.shape[0]`).
3. Create `n` random values in `[0,1)` and save as a new column.
4. Use `.query("column < cutoff")` and `.query("column >= cutoff")` so every row belongs to exactly one group.
5. Compute `len(group)/n*100` and verify the groups cover all rows.

**Key translation:** “roughly 40%” → threshold `0.4`; not exactly 40% because values are random.

In [ ]:
import pandas as pd
import numpy as np

# QUIZ TEMPLATE (uncomment and change filename)
# df = pd.read_csv("quiz_data/fifa23_players_basic.csv")
# n = len(df)                  # same as df.shape[0]
# df["random_var"] = np.random.uniform(size=n)
# groupA = df.query("random_var < 0.4")
# groupB = df.query("random_var >= 0.4")
# print(f"Group A: {len(groupA)/n*100:.2f}%")
# print(f"Group B: {len(groupB)/n*100:.2f}%")
# assert len(groupA) + len(groupB) == n

# Runnable demonstration without external CSV:
df_demo = pd.DataFrame({"player": ["A", "B", "C", "D", "E"], "rating": [80, 75, 92, 64, 88]})
n = len(df_demo)
df_demo["random_var"] = np.random.uniform(size=n)
groupA = df_demo.query("random_var < 0.4")
groupB = df_demo.query("random_var >= 0.4")
print(df_demo)
print("Group A %:", len(groupA)/n*100, "Group B %:", len(groupB)/n*100)
assert len(groupA) + len(groupB) == n

### A — Syntax decoder and mistakes
- `df["random_var"] = ...` **creates or replaces a column**.
- `np.random.uniform(size=n)` creates `n` random numbers between 0 and 1.
- `.query("random_var < 0.4")` filters **rows** using a condition written as a string.
- `.shape` is a tuple `(rows, columns)`; `.shape[0]` is row count. No parentheses: `df.shape`, not `df.shape()`.
- `.query()` returns a new filtered DataFrame; assign it if you want to keep it.
- The threshold `0.4` belongs in **both** comparisons (`<` and `>=`) to avoid missing or double-counting rows.
- `.2f` in an f-string formats a number to two decimal places; it does not change the stored value.

**CHECK:** `len(groupA)+len(groupB)==n`. Random groups will not always be exactly 40/60.

## B — `.iloc` slicing and `.sort_values()`
**Question pattern:** Select rows 28–32 (zero-based), select columns from a given position, sort by weight from largest to smallest.

### WHAT TO DO
1. Read the CSV.
2. Convert the **last desired row index** to an exclusive slice end: `28:33` selects 28,29,30,31,32.
3. Determine the first column position. The practice solution uses `4:` because the desired columns start at position 4 in that file.
4. Sort with `ascending=False` for descending order.

**Memory aid:** `.iloc[ROW_POSITIONS, COLUMN_POSITIONS]` uses integer positions; the end of each slice is excluded.

In [ ]:
# PRACTICE QUIZ TEMPLATE:
# df_cars = pd.read_csv("quiz_data/features.csv")
# df_subset = df_cars.iloc[28:33, 4:]
# df_sorted = df_subset.sort_values(by="weight", ascending=False)
# display(df_sorted)

# Runnable small example:
cars = pd.DataFrame({"model": ["A", "B", "C", "D", "E"],
                     "weight": [1200, 1500, 900, 1800, 1100],
                     "acceleration": [10, 8, 12, 7, 11],
                     "vehicle_id": [101, 102, 103, 104, 105]})
subset = cars.iloc[1:4, 1:4]    # rows 1,2,3; columns 1,2,3
sorted_subset = subset.sort_values(by="weight", ascending=False)
print(sorted_subset)

### B — `.loc` vs `.iloc` and selection cheat sheet
| Goal | Syntax | Important detail |
|---|---|---|
| One column (Series) | `df["weight"]` | One pair of brackets |
| Several columns (DataFrame) | `df[["weight", "vehicle_id"]]` | Inner brackets are a list |
| Row 3, column 0 by position | `df.iloc[3, 0]` | Fourth row, first column |
| Rows 3–7 by position | `df.iloc[3:8, :]` | `8` excluded |
| Specific nonconsecutive rows | `df.iloc[[0, 4, 9], :]` | List of positions |
| Label-based selection | `df.loc[row_label, "weight"]` | Uses labels, not positions |
| Label-based range | `df.loc["a":"d", :]` | End label generally included |
| Heaviest first | `df.sort_values(by="weight", ascending=False)` | Descending |
| Lightest first | `df.sort_values(by="weight")` | Ascending by default |

**CHECK:** `df.columns.tolist()` shows exact names and order. `df_subset.shape` confirms selected dimensions. Sorting returns a new DataFrame unless assigned or `inplace=True` is used. Sorting preserves existing row index labels unless you reset them.

**Common error:** `df.iloc[28:32]` misses row 32. Use `28:33`.

## C — Transform columns using `.apply()`
**Question pattern:** Convert mph to m/s using a function; compute kinetic energy `0.5 × mass × velocity²`; add a new column.

### WHAT TO DO
1. Read the CSV and inspect column names.
2. Write a function taking **one cell value** as input and returning its converted value.
3. Apply that function to the relevant Series: `df["Speed (mph)"].apply(convert_speed)`.
4. Use the resulting Series in an elementwise arithmetic expression to create a new column.

**Key idea:** `.apply(function)` calls the function on **each value** of the Series, producing a new Series. Do **not** write `.apply(convert_speed())` because that calls the function immediately.

In [ ]:
# PRACTICE QUIZ TEMPLATE:
# df_speed = pd.read_csv("quiz_data/speed_mass.csv")

def convert_speed(mph):
    mps = 1609.34 * mph / 3600
    return mps

# mps_series = df_speed["Speed (mph)"].apply(convert_speed)
# df_speed["Kinetic Energy (J)"] = 0.5 * df_speed["Mass (kg)"] * mps_series ** 2

# Runnable demonstration:
df_speed = pd.DataFrame({"Speed (mph)": [0, 30, 60], "Mass (kg)": [1000, 1200, 1500]})
mps_series = df_speed["Speed (mph)"].apply(convert_speed)
df_speed["Kinetic Energy (J)"] = 0.5 * df_speed["Mass (kg)"] * mps_series ** 2
print(df_speed)

### C — Frequently confused operations
- `return mps` **sends a value back** from a function. `print(mps)` only displays it; it does not provide the value `.apply()` needs.
- `df["Mass (kg)"] * mps_series**2` multiplies **corresponding rows**, not entire lists against each other.
- `**2` is exponentiation; `^2` is **not** Python exponentiation.
- A new column is created with `df["new name"] = expression`.
- If `.apply()` is required, show it even though arithmetic directly on a Series can also work.
- `.apply(lambda x: x * factor)` is an alternative to defining a named function, but use the requested form.

**CHECK:** At 0 mph, kinetic energy should be 0. For positive mass, doubling speed should multiply kinetic energy by 4.

### Pandas mini-reference
`df.head()` first rows · `df.columns` column names · `df.shape` dimensions · `df["col"].mean()` mean · `df["col"].nunique()` distinct count · `df.query("col > 10")` filtered rows · `df.reset_index(drop=True)` new sequential index (returned, not automatically saved).

---
# D — Add a method that changes an existing object
**Question pattern:** A `Runner` class is supplied. Add `log_run(self, score)` that checks score, increments `num_runs`, and increases stamina without exceeding 100.

### FIRST: Decode the English into Python
| Instruction | Code pattern |
|---|---|
| “Add a method” | `def log_run(self, score):` **inside** the class |
| “Score must be 0 to 100” | `assert 0 <= score <= 100, "..."` |
| “Increase run count by 1” | `self.num_runs += 1` |
| “Increase stamina by score/10” | `self.stamina + score/10` |
| “Never exceed 100” | `min(calculated_value, 100)` |
| “Use the setter” | `self.set_stamina(new_stamina)` |

**Before coding:** `self` means *the particular Runner object that called the method*. `ana.log_run(80)` uses `self = ana` and `score = 80`.

In [ ]:
class Runner:
    def __init__(self, name, stamina):
        self.name = name
        self.stamina = stamina
        self.num_runs = 0
        self.pace_buddy = None

    def __str__(self):
        return f"{self.name} (stamina {round(self.stamina, 1)})"

    def get_name(self):
        return self.name

    def get_stamina(self):
        return self.stamina

    def get_num_runs(self):
        return self.num_runs

    def get_pace_buddy(self):
        return self.pace_buddy

    def set_stamina(self, value):
        assert 0 <= value <= 100, "stamina must be between 0 and 100"
        self.stamina = value

    def set_pace_buddy(self, other):
        self.pace_buddy = other
        other.pace_buddy = self

    def run_value(self, score):
        return score

    # THE METHOD YOU WOULD ADD IN QUESTION D:
    def log_run(self, score):
        assert 0 <= score <= 100, "score must be between 0 and 100"
        self.num_runs += 1
        new_stamina = min(self.stamina + score / 10, 100)
        self.set_stamina(new_stamina)

ana = Runner("Ana", 50)
ana.log_run(80)
print(ana.get_stamina(), ana.get_num_runs())  # 58.0, 1
ben = Runner("Ben", 95)
ben.log_run(80)
print(ben.get_stamina())                  # 100

### D — Trace and pitfalls
For `ana = Runner("Ana", 50)`:
1. `__init__` creates `name="Ana"`, `stamina=50`, `num_runs=0`, `pace_buddy=None`.
2. `ana.log_run(80)` makes `self` refer to Ana.
3. Assert passes (`80` is in range).
4. `num_runs` changes `0 → 1`.
5. `min(50 + 80/10, 100) = min(58, 100) = 58`.
6. Setter stores stamina 58. No return is required because this method **mutates the object**.

**If stamina starts at 95:** `min(95 + 8, 100) = 100`.

**Common mistakes:** `num_runs += 1` without `self.` inside a method; `self.stamina = ...` despite instructions to use setter; `return self.set_stamina(...)` when no return requested; defining method outside the class; `assert` with the wrong inequality.

**General method template:**
```python
def method_name(self, input_value):
    assert CONDITION, "helpful message"
    self.counter += 1
    new_value = min(self.attribute + input_value, LIMIT)
    self.set_attribute(new_value)
```

**Remember:** getter reads, setter updates, `assert` validates, `return` outputs, `print` displays.

---
# E — Inheritance, overriding, and class methods
**Question pattern:** Write `TrailRunner(Runner)` with a shared multiplier, modified stamina limit, modified run score, and a class method that changes the multiplier.

### FIRST: Translate the instructions
1. “Inherits from Runner” → `class TrailRunner(Runner):`.
2. “Shared multiplier 1.25” → class variable **outside** `__init__`: `backpack_multiplier = 1.25`.
3. “Overrides setter” → write `def set_stamina(self, value)` with the **same name** as parent method; child behavior takes precedence.
4. “Overrides run_value” → write `def run_value(self, score)` with the new formula.
5. “Class method” → put `@classmethod` immediately above `def set_backpack_multiplier(cls, value)`.

**Important:** You do **not** need to write a new `__init__` if the inherited one already creates the needed attributes.

In [ ]:
class TrailRunner(Runner):
    backpack_multiplier = 1.25  # CLASS variable, shared through the class

    def set_stamina(self, value):  # overrides Runner.set_stamina
        assert 0 <= value <= self.backpack_multiplier * 100, "stamina out of range"
        self.stamina = value

    def run_value(self, score):    # overrides Runner.run_value
        return score * self.backpack_multiplier

    @classmethod
    def set_backpack_multiplier(cls, value):
        assert 1 <= value <= 2, "multiplier must be between 1 and 2"
        cls.backpack_multiplier = value

regular = Runner("Dee", 70)
trail = TrailRunner("Cy", 70)
print(regular.run_value(60))  # 60
print(trail.run_value(60))    # 75.0
trail.set_stamina(120)       # allowed: max is 125
TrailRunner.set_backpack_multiplier(1.5)
print(trail.run_value(60))    # 90.0; existing object sees new class variable
trail.set_stamina(140)       # allowed: max is now 150
TrailRunner.set_backpack_multiplier(1.25)  # restore for later examples

### E — Why each part works
**Parent vs child:** `Runner` defines default behavior; `TrailRunner` inherits the constructor and getters, but overrides two methods. `Runner("Dee",70).run_value(60)` returns 60; `TrailRunner("Cy",70).run_value(60)` returns 75.

**Class variable vs instance attribute:** `backpack_multiplier = 1.25` belongs to the class; `self.stamina` belongs to one object. Updating `TrailRunner.backpack_multiplier` affects instances that look up the shared class variable. `self.backpack_multiplier` reads it (unless the instance has its own shadowing attribute).

**`self` vs `cls`:** ordinary instance methods receive an **object** as `self`; class methods receive the **class** as `cls`. `cls.backpack_multiplier = value` updates the shared class variable. The decorator `@classmethod` is required.

**Why `self.backpack_multiplier` instead of hard-coding `TrailRunner.backpack_multiplier`?** It follows the object's class lookup and is more reusable, including for subclasses. The supplied practice solution uses this style.

**Checks:** `TrailRunner.set_backpack_multiplier(0.8)` should fail; `2.5` should fail; exactly `1` and `2` should pass. At multiplier 1.5, stamina 150 is allowed, 160 is not.

**Important subtlety:** `Runner.log_run` from D, if inherited unchanged, still caps the new stamina at 100 because its implementation explicitly uses `min(...,100)`. Overriding `set_stamina` alone does not rewrite that formula. This is not a requirement of question E; it illustrates how inherited methods behave.

### E — Inheritance template for a different quiz problem
```python
class Child(Parent):
    shared_factor = 1.25

    def existing_method(self, value):  # override
        return value * self.shared_factor

    @classmethod
    def set_shared_factor(cls, value):
        assert LOWER <= value <= UPPER, "invalid factor"
        cls.shared_factor = value
```

**Ask yourself:** Which methods already exist in the parent? Which are explicitly overridden? Is the variable shared by all objects, or unique to each? Does the question require `return` or mutation?

---
# F — Composition: one class holds a list of other objects
**Question pattern:** `RunClub` receives a dictionary, creates `Runner` objects, stores them in a list, and pairs runners with the first eligible available partner.

### PART 1: Translate the constructor
Input: `{"Ana": 50, "Ben": 95, "Cy": 56, "Dee": 47}`.

- `roster` is a **dictionary**: names are keys; stamina numbers are values.
- `for name in roster.keys()` produces `"Ana"`, then `"Ben"`, etc.
- `roster[name]` looks up that name's stamina (e.g. `roster["Ana"]` is 50).
- `Runner(name, roster[name])` **creates a new Runner object**.
- `self.runners.append(runner)` stores that object in the club's list.

**Do not confuse:** `self.runners` is a list of **Runner objects**, not a list of strings or stamina numbers.

In [ ]:
class RunClub:
    max_gap = 10

    def __init__(self, roster):
        self.runners = []
        for name in roster.keys():
            runner = Runner(name, roster[name])
            self.runners.append(runner)

    def swoop_set_buddys(self):  # keep the exact spelling required by the quiz
        n = len(self.runners)
        for i in range(n):
            runner_i = self.runners[i]
            stam_i = runner_i.get_stamina()

            if runner_i.get_pace_buddy() is not None:
                continue  # runner already paired: go to next i

            for j in range(i + 1, n):
                runner_j = self.runners[j]
                stam_j = runner_j.get_stamina()

                if runner_j.get_pace_buddy() is not None:
                    continue  # candidate already paired: try next j

                if abs(stam_j - stam_i) <= self.max_gap:
                    runner_i.set_pace_buddy(runner_j)
                    break  # first eligible match found; stop inner loop

club = RunClub({"Ana": 50, "Ben": 95, "Cy": 56, "Dee": 47})
club.swoop_set_buddys()
for runner in club.runners:
    print(runner, "is paired with", runner.get_pace_buddy())

### F — PART 2: Trace the nested loops (do this on scrap paper if stuck)
List positions: `0=Ana(50)`, `1=Ben(95)`, `2=Cy(56)`, `3=Dee(47)`.

| Outer `i` | Candidate `j` | Check | Outcome |
|---|---|---|---|
| 0 Ana | 1 Ben | `abs(50-95)=45 > 10` | No pair |
| 0 Ana | 2 Cy | `abs(50-56)=6 <= 10` | **Pair Ana–Cy**, then `break` |
| 1 Ben | 2 Cy | Cy already paired | `continue` inner loop |
| 1 Ben | 3 Dee | `abs(95-47)=48 > 10` | No pair |
| 2 Cy | — | Cy already paired | `continue` outer loop |
| 3 Dee | — | `range(4,4)` is empty | No pair |

**Final:** Ana ↔ Cy; Ben and Dee unpaired.

**Why `range(i+1,n)`?** It only examines runners **after** the current runner, so it never checks the same runner or earlier runners. This is sufficient for this first-available matching procedure: if an earlier runner could have paired with the current runner, it would already have done so when its turn came.

**Why `continue`?** Skip a runner/candidate who is already paired. **Why `break`?** Stop searching once the first eligible candidate is found. `break` exits only the **inner** loop here.

**Why `abs(...)`?** The stamina difference should be positive regardless of which runner has higher stamina. `abs(50-56)=6`.

**Why no return?** `set_pace_buddy` mutates both Runner objects. `swoop_set_buddys` changes their state, not a value to return.

### F — PART 3: The key object-reference idea
The supplied `Runner` method is:
```python
def set_pace_buddy(self, other):
    self.pace_buddy = other
    other.pace_buddy = self
```
Calling `ana.set_pace_buddy(cy)` stores a **reference to Cy's object** inside Ana and a **reference to Ana's object** inside Cy. It does not store just their names.

That is why `runner.get_pace_buddy()` returns either `None` (unpaired) or another `Runner` object. When printing, Python uses that object's `__str__` method.

**Don't accidentally use:** `runner_i.pace_buddy = runner_j` if the question explicitly says to use `set_pace_buddy`—the setter updates **both** sides.

**Common errors:** `for j in range(n)` without guarding against self-pairing; forgetting `break`; checking only the candidate's buddy and not the current runner's; comparing raw signed differences without `abs`; pairing runners whose difference is **strictly less than** 10 when exactly 10 should be allowed; returning a list when the method is supposed to mutate objects.

**Alternative nested-loop template** (directly loop over objects):
```python
for runner in self.runners:
    if runner.get_pace_buddy() is not None:
        continue
    for other in self.runners:
        if runner is other or other.get_pace_buddy() is not None:
            continue
        if abs(runner.get_stamina()-other.get_stamina()) <= self.max_gap:
            runner.set_pace_buddy(other)
            break
```
The **indexed version above** mirrors practice Solution 1 and avoids checking earlier positions. Both are valid when correctly guarded.

---
# Emergency reference — What do I write first?
| If the question says... | First code to write |
|---|---|
| “Read a CSV” | `df = pd.read_csv("path/file.csv")` |
| “Number of rows” | `n = len(df)` or `df.shape[0]` |
| “New column” | `df["new"] = ...` |
| “Filter rows” | `subset = df.query("condition")` |
| “Rows 28 through 32” | `df.iloc[28:33, :]` |
| “Columns by name” | `df[["col1", "col2"]]` |
| “Sort descending” | `df.sort_values(by="col", ascending=False)` |
| “Apply a function to each value” | `df["col"].apply(function_name)` |
| “Define a class” | `class Name:` |
| “Create instance attributes” | `def __init__(self, ...): self.attr = ...` |
| “Shared class variable” | `class Name: shared = VALUE` |
| “Inherit from Parent” | `class Child(Parent):` |
| “Override a method” | `def same_method_name(self, ...):` inside child |
| “Class method” | `@classmethod` then `def method(cls, ...):` |
| “Validate input” | `assert condition, "message"` |
| “Change object state” | `self.attr = ...` or call required setter |
| “Return a calculated value” | `return expression` |
| “Create objects from dictionary” | `for key in d: objects.append(Class(key,d[key]))` |
| “First eligible match” | nested loops + condition + `break` |
| “Skip already used” | `if used: continue` |

## Final debugging checklist
1. **Exact names:** Class, method, parameter, and column names match the prompt (including `swoop_set_buddys`).
2. **Indentation:** Methods inside class; conditions inside correct loops; `break` inside inner loop.
3. **`self`/`cls`:** Use `self` for instance state and methods; `cls` with `@classmethod`.
4. **Return vs mutation:** A function needing a result must `return`; an update method may have no return.
5. **Objects vs values:** `Runner(...)` constructs an object; `runner.get_stamina()` gets its number.
6. **Bounds:** `<=` versus `<`, `min` cap, and `.iloc` exclusive end.
7. **Test:** Run supplied test cell; inspect output; test boundary values and invalid inputs when asked.
8. **Notebook execution order:** Run setup cells before dependent cells; if you redefine a class, rerun cells that create its instances.

### Quick recovery when panicking
**Read one bullet of the prompt → translate it to one code line → check it off → proceed.** You do not need to understand the whole program at once. For OOP, first identify what the object stores, then what each method reads, changes, or returns.

*Source basis: supplied Quiz 4 practice questions and solutions, organized A–F. General debugging and alternate templates are supplementary explanations.*